<a href="https://colab.research.google.com/github/bahmedx/730/blob/main/Bayesian_Inference_Enhancing_a_Medical_Diagnosis_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1: Install & Import Libraries**

In [ ]:
# ==================================================
# Step 1: Install & Import Libraries
# ==================================================

!pip install -q pymc arviz seaborn scikit-learn

import os
import zipfile
import warnings

import numpy as np
import pandas as pd

import pymc as pm
import arviz as az

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    log_loss
)

warnings.filterwarnings("ignore")

np.random.seed(42)

print("Libraries loaded successfully.")

# **Step 2: Upload & Extract Dataset**

In [ ]:
# ==================================================
# Step 2: Upload & Extract Dataset
# ==================================================

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

extract_dir = "heart_data"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted successfully.")

# **Step 3: Load Datasets**

In [ ]:
# ==================================================
# Step 3: Load Datasets
# ==================================================

columns = [
    'age','sex','cp','trestbps','chol',
    'fbs','restecg','thalach','exang',
    'oldpeak','slope','ca','thal','num'
]

data_files = [
    'processed.cleveland.data',
    'reprocessed.hungarian.data',
    'processed.switzerland.data',
    'processed.va.data'
]

datasets = {}

dataset_names = {
    'processed.cleveland.data': 'Cleveland',
    'reprocessed.hungarian.data': 'Hungary',
    'processed.switzerland.data': 'Switzerland',
    'processed.va.data': 'VA'
}

datasets = {}

for file in data_files:

    for root, dirs, files_in_dir in os.walk(extract_dir):

        if file in files_in_dir:

            path = os.path.join(root, file)

            df = pd.read_csv(
                path,
                names=columns,
                na_values='?',
                sep=r'\s*,\s*|\s+',
                engine='python'
            )

            friendly_name = dataset_names[file]
            df['dataset'] = friendly_name
            datasets[friendly_name] = df
            print(
                f"{friendly_name}: {df.shape[0]} rows"
            )

print("\nDatasets loaded successfully.")

# **Step 4: Data Preprocessing**

In [ ]:
# ==================================================
# Step 4: Data Preprocessing
# ==================================================

combined_df = pd.concat(
    datasets.values(),
    ignore_index=True
)

print(
    "Combined Shape:",
    combined_df.shape
)

# Convert all variables to numeric

for col in columns:

    combined_df[col] = pd.to_numeric(
        combined_df[col],
        errors="coerce"
    )

# Replace UCI missing value codes
combined_df.replace(-9, np.nan, inplace=True)

# Drop rows where the target column is invalid
combined_df = combined_df.dropna(subset=['num'])

# Impute missing values with column medians
combined_df = combined_df.fillna(combined_df.median(numeric_only=True))

# Remove duplicates
combined_df = combined_df.drop_duplicates()

print(
    "Final Shape:",
    combined_df.shape
)

# Separate features (X) and binarize target (y: 0 = healthy, 1 = disease)
# X = combined_df.drop('num', axis=1)
y = (combined_df['num'] > 0).astype(int)

# Store Dataset labels
dataset_labels = combined_df["dataset"]

# Create feature set
X = combined_df.drop(
    ["num"],
    axis=1
)

# Drop the 'dataset' column from X as it is a string and not a feature for the model
#X = X.drop('dataset', axis=1)


# Categorical features

categorical_cols = [
    "cp",
    "restecg",
    "slope",
    "thal",
    "dataset"
]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

print("Features:", X.shape)
print("Target:", y.shape)


# **Step 5: Exploratory Data Analysis**

In [ ]:
# ==================================================
# Step 5: Exploratory Data Analysis
# ==================================================

prevalence = pd.DataFrame({
    "Dataset": dataset_labels,
    "Disease": y
})

prevalence = (
    prevalence
    .groupby("Dataset")
    .mean()
    .reset_index()
)

dataset_sizes = (
    combined_df["dataset"]
    .value_counts()
    .reset_index()
)

dataset_sizes.columns = [
    "Dataset",
    "Patients"
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4)
)

# -------------------------------------
# Disease Prevalence
# -------------------------------------

sns.barplot(
    data=prevalence,
    x="Dataset",
    y="Disease",
    ax=axes[0],
    palette="Blues"
)

axes[0].set_title(
    "Heart Disease Prevalence"
)

axes[0].set_ylabel(
    "Prevalence"
)

for container in axes[0].containers:
    axes[0].bar_label(
        container,
        fmt="%.2f"
    )

# -------------------------------------
# Dataset Size
# -------------------------------------

sns.barplot(
    data=dataset_sizes,
    x="Dataset",
    y="Patients",
    ax=axes[1],
    palette="Greens"
)

axes[1].set_title(
    "Patients per Dataset"
)

for container in axes[1].containers:
    axes[1].bar_label(
        container,
        fmt="%d"
    )

plt.suptitle(
    "Exploratory Dataset Overview",
    fontsize=14
)

plt.tight_layout()

plt.show()

# **Step 6: Train Combined Bayesian Model**

In [ ]:
# ==================================================
# Step 6: Train Combined Bayesian Model
# ==================================================

# Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

dataset_test = dataset_labels.loc[
    X_test.index
]

# Scale Features

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

# Bayesian Logistic Regression

coords = {
    "predictor": X.columns
}

with pm.Model(coords=coords) as heart_model:

    alpha = pm.Normal(
        "alpha",
        mu=0,
        sigma=2
    )

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=2,
        dims="predictor"
    )

    mu = alpha + pm.math.dot(
        X_train_scaled,
        beta
    )

    p = pm.math.sigmoid(mu)

    y_obs = pm.Bernoulli(
        "y_obs",
        p=p,
        observed=y_train.values
    )

    trace = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.90,
        random_seed=42,
        return_inferencedata=True
    )

print("Bayesian model trained.")

# **Step 7: Evaluate Combined Bayesian Model**

In [ ]:
# ==================================================
# Step 7: Evaluate Combined Bayesian Model
# ==================================================

alpha = (
    trace.posterior["alpha"]
    .mean()
    .values
)

beta = (
    trace.posterior["beta"]
    .mean(dim=["chain", "draw"])
    .values
)

logits = (
    alpha +
    np.dot(
        X_test_scaled,
        beta
    )
)

probs = (
    1 /
    (1 + np.exp(-logits))
)

preds = (
    probs >= 0.5
).astype(int)

# Metrics

accuracy = accuracy_score(
    y_test,
    preds
)

auc = roc_auc_score(
    y_test,
    probs
)

ll = -log_loss(
    y_test,
    probs
)

print(
    f"Accuracy: {accuracy:.4f}"
)

print(
    f"ROC-AUC: {auc:.4f}"
)

print(
    f"Log-Likelihood: {ll:.4f}"
)

### **Step 7A: Visual Evaluation (Confusion Matrix + ROC Curve)**

In [ ]:
# ==================================================
# Step 7A: Visual Evaluation
# ==================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12,5)
)

# ---------------------------------------
# Confusion Matrix
# ---------------------------------------

cm = confusion_matrix(
    y_test,
    preds
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
    ax=axes[0]
)

axes[0].set_title(
    "Confusion Matrix"
)

axes[0].set_xlabel(
    "Predicted"
)

axes[0].set_ylabel(
    "Actual"
)

# ---------------------------------------
# ROC Curve
# ---------------------------------------

fpr, tpr, _ = roc_curve(
    y_test,
    probs
)

axes[1].plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"AUC = {auc:.3f}"
)

axes[1].plot(
    [0,1],
    [0,1],
    "--",
    color="gray"
)

axes[1].legend(
    loc="lower right"
)

axes[1].set_title(
    "ROC Curve"
)

axes[1].set_xlabel(
    "False Positive Rate"
)

axes[1].set_ylabel(
    "True Positive Rate"
)

plt.suptitle(
    "Combined Bayesian Model Evaluation",
    fontsize=14
)

plt.tight_layout()

plt.show()

# **Step 8: Dataset-Specific Predictions**

In [ ]:
# ==================================================
# Step 8: Dataset-Specific Predictions
# ==================================================

dataset_results = []

for ds in dataset_test.unique():

    mask = (
        dataset_test == ds
    )

    if len(
        np.unique(y_test[mask])
    ) > 1:

        roc_auc = roc_auc_score(
            y_test[mask],
            probs[mask]
        )

    else:

        roc_auc = np.nan

    dataset_results.append({

        "Dataset": ds,

        "Patients":
        mask.sum(),

        "Accuracy":
        accuracy_score(
            y_test[mask],
            preds[mask]
        ),

        "ROC_AUC":
        roc_auc,

        "Mean Risk":
        probs[mask].mean()

    })

dataset_results = pd.DataFrame(
    dataset_results
)

display(dataset_results)

# **Step 9: Dataset Comparison Dashboard**

In [ ]:
# ==================================================
# Step 9: Compare Heart Disease Across Datasets
# ==================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10)
)

# ----------------------------------------
# 1. Dataset Accuracy
# ----------------------------------------

sns.barplot(
    data=dataset_results,
    x="Dataset",
    y="Accuracy",
    palette="Blues",
    ax=axes[0, 0]
)

axes[0, 0].set_title(
    "Prediction Accuracy by Dataset"
)

axes[0, 0].set_ylabel(
    "Accuracy"
)

axes[0, 0].set_ylim(0, 1)

for container in axes[0, 0].containers:
    axes[0, 0].bar_label(
        container,
        fmt="%.2f"
    )

# ----------------------------------------
# 2. ROC-AUC
# ----------------------------------------

roc_plot = dataset_results.copy()

# Replace NaN for plotting only
roc_plot["ROC_AUC"] = (
    roc_plot["ROC_AUC"]
    .fillna(0)
)

sns.barplot(
    data=roc_plot,
    x="Dataset",
    y="ROC_AUC",
    palette="Greens",
    ax=axes[0, 1]
)

axes[0, 1].set_title(
    "ROC-AUC by Dataset"
)

axes[0, 1].set_ylabel(
    "ROC-AUC"
)

axes[0, 1].set_ylim(0, 1)

for container in axes[0, 1].containers:
    axes[0, 1].bar_label(
        container,
        fmt="%.2f"
    )

# ----------------------------------------
# 3. Mean Posterior Risk
# ----------------------------------------

sns.barplot(
    data=dataset_results,
    x="Dataset",
    y="Mean Risk",
    palette="Oranges",
    ax=axes[1, 0]
)

axes[1, 0].set_title(
    "Mean Posterior Heart Disease Risk"
)

axes[1, 0].set_ylabel(
    "Average Probability"
)

axes[1, 0].set_ylim(
    0,
    dataset_results["Mean Risk"].max() * 1.15
)

for container in axes[1, 0].containers:
    axes[1, 0].bar_label(
        container,
        fmt="%.2f"
    )

# ----------------------------------------
# 4. Number of Patients
# ----------------------------------------

sns.barplot(
    data=dataset_results,
    x="Dataset",
    y="Patients",
    palette="Purples",
    ax=axes[1, 1]
)

axes[1, 1].set_title(
    "Patients Evaluated"
)

axes[1, 1].set_ylabel(
    "Patients"
)

for container in axes[1, 1].containers:
    axes[1, 1].bar_label(
        container,
        fmt="%d"
    )

# ----------------------------------------
# Formatting
# ----------------------------------------

for ax in axes.flat:

    ax.tick_params(
        axis="x",
        rotation=25
    )

plt.suptitle(
    "Dataset Comparison Dashboard",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()

plt.show()

# **Step 10: Dynamic Bayesian Updating**

In [ ]:
# ==================================================
# Step 10: Dynamic Bayesian Updating
# ==================================================

# Simulated new patient observations

new_cases = X_test_scaled[:25]

new_labels = y_test.iloc[:25]

# Update training set

X_updated = np.vstack([
    X_train_scaled,
    new_cases
])

y_updated = pd.concat([
    y_train,
    new_labels
])

with pm.Model(coords=coords) as updated_model:

    alpha = pm.Normal(
        "alpha",
        mu=0,
        sigma=2
    )

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=2,
        dims="predictor"
    )

    mu = alpha + pm.math.dot(
        X_updated,
        beta
    )

    p = pm.math.sigmoid(mu)

    y_obs = pm.Bernoulli(
        "y_obs",
        p=p,
        observed=y_updated.values
    )

    updated_trace = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.90,
        random_seed=42,
        return_inferencedata=True
    )

print("Posterior updated.")

### **Step 10A: Evaluate Updated Model**

In [ ]:
# ==================================================
# Step 10A: Updated Model Evaluation
# ==================================================

updated_alpha = (
    updated_trace.posterior["alpha"]
    .mean()
    .values
)

updated_beta = (
    updated_trace.posterior["beta"]
    .mean(dim=["chain", "draw"])
    .values
)

updated_logits = (
    updated_alpha +
    np.dot(
        X_test_scaled,
        updated_beta
    )
)

updated_probs = (
    1 /
    (1 + np.exp(-updated_logits))
)

updated_preds = (
    updated_probs >= 0.5
).astype(int)

updated_accuracy = accuracy_score(
    y_test,
    updated_preds
)

updated_auc = roc_auc_score(
    y_test,
    updated_probs
)

updated_ll = -log_loss(
    y_test,
    updated_probs
)

# **Step 11: Compare Original vs Updated Models**

In [ ]:
# ==================================================
# Step 11: Compare Original vs Updated
# ==================================================

comparison = pd.DataFrame({

    "Model": [
        "Original",
        "Updated"
    ],

    "Accuracy": [
        accuracy,
        updated_accuracy
    ],

    "ROC_AUC": [
        auc,
        updated_auc
    ],

    "Log_Likelihood": [
        ll,
        updated_ll
    ]

})

display(comparison)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14,4)
)

colors = [
    "steelblue",
    "darkorange"
]

# Accuracy

axes[0].bar(
    comparison["Model"],
    comparison["Accuracy"],
    color=colors
)

axes[0].set_title(
    "Accuracy"
)

for i, v in enumerate(comparison["Accuracy"]):
    axes[0].text(
        i,
        v + 0.01,
        f"{v:.3f}",
        ha="center"
)

# ROC-AUC

axes[1].bar(
    comparison["Model"],
    comparison["ROC_AUC"],
    color=colors
)

axes[1].set_title(
    "ROC-AUC"
)

for i, v in enumerate(comparison["ROC_AUC"]):
    axes[1].text(
        i,
        v + 0.01,
        f"{v:.3f}",
        ha="center"
)

# Log-Likelihood

axes[2].bar(
    comparison["Model"],
    comparison["Log_Likelihood"],
    color=colors
)

axes[2].set_title(
    "Log-Likelihood"
)

for i, v in enumerate(comparison["Log_Likelihood"]):
    axes[2].text(
        i,
        v + 0.01,
        f"{v:.3f}",
        ha="center"
)

plt.suptitle(
    "Original vs Updated Bayesian Model",
    fontsize=14
)

plt.tight_layout()

plt.show()

# **Step 12: Train Dataset-Specific Bayesian Models**

In [ ]:
# ==================================================
# Step 12: Train Dataset-Specific Bayesian Models
# ==================================================

dataset_models = {}

for dataset_name in datasets.keys():

    print(f"\nTraining model for {dataset_name}")

    df_ds = datasets[dataset_name].copy()

    # ------------------------------
    # Convert to numeric
    # ------------------------------

    for col in columns:

        if col != "dataset":

            df_ds[col] = pd.to_numeric(
                df_ds[col],
                errors="coerce"
            )

    # ------------------------------
    # Clean data
    # ------------------------------
    """
    df_ds.replace(
        -9,
        np.nan,
        inplace=True
    )
    """
    df_ds = df_ds.dropna(
        subset=["num"]
    )

    df_ds = df_ds.fillna(
        df_ds.median(
            numeric_only=True
        )
    )

    # ------------------------------
    # Target
    # ------------------------------

    y_ds = (
        df_ds["num"] > 0
    ).astype(int)

    # ------------------------------
    # Features
    # ------------------------------

    X_ds = df_ds.drop(
        ["num", "dataset"],
        axis=1,
        errors="ignore"
    )

    X_ds = pd.get_dummies(
        X_ds,
        columns=[
            "cp",
            "restecg",
            "slope",
            "thal"
        ],
        drop_first=True
    )

    X_ds = X_ds.reindex(
        columns=X.columns,
        fill_value=0
    )

    scaler_ds = StandardScaler()

    X_ds_scaled = (
        scaler_ds.fit_transform(X_ds)
    )

    coords_ds = {
        "predictor": X_ds.columns
    }

    with pm.Model(
        coords=coords_ds
    ) as model_ds:

        alpha = pm.Normal(
            "alpha",
            mu=0,
            sigma=2
        )

        beta = pm.Normal(
            "beta",
            mu=0,
            sigma=2,
            dims="predictor"
        )

        mu = (
            alpha
            + pm.math.dot(
                X_ds_scaled,
                beta
            )
        )

        p = pm.math.sigmoid(mu)

        pm.Bernoulli(
            "y_obs",
            p=p,
            observed=y_ds.values
        )

        trace_ds = pm.sample(
            draws=500,
            tune=500,
            chains=2,
            target_accept=0.90,
            progressbar=False,
            random_seed=42,
            return_inferencedata=True
        )

    dataset_models[
        dataset_name
    ] = {

        "trace": trace_ds,
        "scaler": scaler_ds,
        "columns": X.columns

    }

print(
    "\nAll dataset models trained."
)

# **Step 13: Patient Prediction Function**

In [ ]:
# ==================================================
# Step 13: Patient Prediction Function
# ==================================================

def predict_patient_risk(patient_df, model_info):

    patient_df = patient_df.copy()

    patient_df = pd.get_dummies(
        patient_df,
        columns=["cp", "restecg", "slope", "thal"],
        drop_first=False
    )

    patient_df = patient_df.reindex(
        columns=model_info["columns"],
        fill_value=0
    )

    patient_scaled = model_info["scaler"].transform(patient_df)[0]

    trace = model_info["trace"]

    alpha_samples = (
        trace.posterior["alpha"]
        .stack(sample=("chain", "draw"))
        .values
    )

    beta_samples = (
        trace.posterior["beta"]
        .stack(sample=("chain", "draw"))
        .transpose("sample", "predictor")
        .values
    )

    logits = (
        alpha_samples
        + np.dot(
            beta_samples,
            patient_scaled
        )
    )
    # logits = alpha + beta @ patient_scaled

    probabilities = 1 / (1 + np.exp(-logits))

    mean_risk = probabilities.mean()

    lower_bound = np.percentile(
        probabilities,
        2.5
    )

    upper_bound = np.percentile(
        probabilities,
        97.5
    )

    return (
        mean_risk,
        lower_bound,
        upper_bound
    )


print(
    "True Bayesian prediction function created."
)


# **Step 14: Create New Patient**

In [ ]:
# ==================================================
# Step 14: Create New Patient
# ==================================================

patient = pd.DataFrame([{

    "age":60,
    "sex":1,
    "cp":3,
    "trestbps":145,
    "chol":250,
    "fbs":0,
    "restecg":1,
    "thalach":130,
    "exang":1,
    "oldpeak":2.3,
    "slope":2,
    "ca":1,
    "thal":3

}])

print(
    "Patient Being Evaluated"
)

display(patient)

print(
    "\nTraining Data Summary"
)

display(
    combined_df[
        [
            "age",
            "chol",
            "thalach",
            "oldpeak",
            "ca"
        ]
    ].describe()
)

# **Step 15: Predict Heart Disease Risk**

In [ ]:
# ==================================================
# Step 15: Predict Heart Disease Risk
# ==================================================

prediction_results = []

# -----------------------------------
# Dataset Models
# -----------------------------------

for dataset_name, model_info in dataset_models.items():

    risk, lower, upper = (
        predict_patient_risk(
            patient,
            model_info
        )
    )

    prediction_results.append({

        "Dataset":
        dataset_name,

        "Heart Disease Risk":
        risk,

        "Lower 95% CI":
        lower,

        "Upper 95% CI":
        upper

    })

# -----------------------------------
# Combined Model
# -----------------------------------

combined_model_info = {

    "trace": trace,
    "scaler": scaler,
    "columns": X.columns

}

combined_risk, combined_lower, combined_upper = (
    predict_patient_risk(
        patient,
        combined_model_info
    )
)

prediction_results.append({

    "Dataset":
    "Combined Model",

    "Heart Disease Risk":
    combined_risk,

    "Lower 95% CI":
    combined_lower,

    "Upper 95% CI":
    combined_upper

})

# -----------------------------------
# Consensus
# -----------------------------------

individual_risks = [

    row["Heart Disease Risk"]

    for row in prediction_results

    if row["Dataset"]
    != "Combined Model"

]

consensus_risk = np.mean(
    individual_risks
)

prediction_results.append({

    "Dataset":
    "Consensus Average",

    "Heart Disease Risk":
    consensus_risk,

    "Lower 95% CI":
    np.nan,

    "Upper 95% CI":
    np.nan

})

prediction_results = pd.DataFrame(
    prediction_results
)

risk_cols = [

    "Heart Disease Risk",
    "Lower 95% CI",
    "Upper 95% CI"

]

prediction_results[
    risk_cols
] = (
    prediction_results[
        risk_cols
    ]
    * 100
).round(2)

display(
    prediction_results
)

### **Step 15A: Patient Risk Summary**

In [ ]:
# ==================================================
# Step 15A: Credible Interval Visualization
# ==================================================

plot_df = prediction_results[
    prediction_results["Dataset"]
    != "Consensus Average"
]

plt.figure(
    figsize=(9,4)
)

plt.errorbar(

    plot_df["Dataset"],

    plot_df["Heart Disease Risk"],

    yerr=[

        plot_df["Heart Disease Risk"]
        - plot_df["Lower 95% CI"],

        plot_df["Upper 95% CI"]
        - plot_df["Heart Disease Risk"]

    ],

    fmt="o",
    capsize=5

)

plt.ylabel(
    "Heart Disease Risk (%)"
)

plt.title(
    "Risk Predictions with 95% Credible Intervals"
)

plt.show()

# **Step 16: Final Patient Assessment**

In [ ]:
# ==================================================
# Step 16: Final Patient Assessment
# ==================================================

consensus_percent = (

    prediction_results.loc[
        prediction_results["Dataset"]
        ==
        "Consensus Average",

        "Heart Disease Risk"
    ]

    .iloc[0]

)

combined_percent = (

    prediction_results.loc[
        prediction_results["Dataset"]
        ==
        "Combined Model",

        "Heart Disease Risk"
    ]

    .iloc[0]

)

if consensus_percent >= 70:

    risk_category = "High Risk"

elif consensus_percent >= 40:

    risk_category = "Moderate Risk"

else:

    risk_category = "Low Risk"

print("=" * 70)

print(
    "FINAL PATIENT HEART DISEASE ASSESSMENT"
)

print("=" * 70)

print(
    f"Combined Model Risk: "
    f"{combined_percent:.2f}%"
)

print(
    f"Consensus Average Risk: "
    f"{consensus_percent:.2f}%"
)

print(
    f"Risk Category: "
    f"{risk_category}"
)

### **Step 16A: Final Dashboard**

In [ ]:
# ==================================================
# Step 16A: Final Dashboard
# ==================================================

risk_colors = [

    "#4E79A7",  # Cleveland
    "#59A14F",  # Hungary
    "#F28E2B",  # Switzerland
    "#E15759",  # VA
    "#AF7AA1",  # Combined
    "#EDC948"   # Consensus

]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14,5)
)

# -----------------------------------------
# All Predictions
# -----------------------------------------

bars = axes[0].bar(
    prediction_results["Dataset"],
    prediction_results["Heart Disease Risk"],
    color=risk_colors
)

axes[0].set_title(
    "Patient Risk Comparison"
)

axes[0].set_ylabel(
    "Heart Disease Risk (%)"
)

axes[0].set_ylim(0,100)

axes[0].tick_params(
    axis="x",
    rotation=30
)

for bar in bars:

    height = bar.get_height()

    axes[0].text(
        bar.get_x()
        + bar.get_width()/2,
        height + 1,
        f"{height:.1f}%",
        ha="center"
    )

# Get combined and consensus percentages
consensus_percent = (

    prediction_results.loc[
        prediction_results["Dataset"]
        ==
        "Consensus Average",

        "Heart Disease Risk"
    ]

    .iloc[0]

)

combined_percent = (

    prediction_results.loc[
        prediction_results["Dataset"]
        ==
        "Combined Model",

        "Heart Disease Risk"
    ]

    .iloc[0]

)

# -----------------------------------------
# Combined vs Consensus
# -----------------------------------------

summary_df = pd.DataFrame({

    "Metric": [
        "Combined Model",
        "Consensus Average"
    ],

    "Risk": [
        combined_percent,
        consensus_percent
    ]

})

bars2 = axes[1].bar(

    summary_df["Metric"],
    summary_df["Risk"],

    color=[
        "#AF7AA1",
        "#EDC948"
    ]

)

axes[1].set_title(
    "Overall Risk Summary"
)

axes[1].set_ylim(0,100)

for bar in bars2:

    height = bar.get_height()

    axes[1].text(
        bar.get_x()
        + bar.get_width()/2,
        height + 1,
        f"{height:.1f}%",
        ha="center"
    )

plt.suptitle(
    "Final Patient Heart Disease Assessment",
    fontsize=14
)

plt.tight_layout()

plt.show()

print(
    "\nNote: 'Combined Model' is "
    "trained on all datasets "
    "simultaneously, while "
    "'Consensus Average' is the "
    "mean prediction from the "
    "four population-specific models."
)